# V2: Tokenizing the Train and Validation Sets
## 2.1 Loading the Train/Validation Sets

In [11]:
from src.v2.tokenize_diff import tokenize_diff
import pandas as pd

train_v2 = pd.read_csv('data/processed/train_v2.csv')
validation_v2 = pd.read_csv('data/processed/validation_v2.csv')

## 2.2 Applying the Tokenizer to Train and Validation Sets

In [12]:
train_v2['tokens'] = train_v2['diff'].apply(tokenize_diff)
validation_v2['tokens'] = validation_v2['diff'].apply(tokenize_diff)

print('Train_V2 Tokenized:')
print(train_v2["diff"][0])
print(train_v2['tokens'][0])
print('\nValidation_V2 Tokenized:')
print(validation_v2['diff'][0])
print(validation_v2['tokens'][0])

Train_V2 Tokenized:
- for i, j in enumerate(itertools.permutations(r, 3)):
+ for i, j in enumerate(itertools.permutations(r)):
['<DELETE>', '<INDENT_0>', 'for', 'i', ',', 'j', 'in', 'enumerate', '(', 'itertools', '.', 'permutations', '(', 'r', ',', '3', ')', ')', ':', '<ADD>', '<INDENT_0>', 'for', 'i', ',', 'j', 'in', 'enumerate', '(', 'itertools', '.', 'permutations', '(', 'r', ')', ')', ':']

Validation_V2 Tokenized:
+ print(ans)
['<ADD>', '<INDENT_0>', 'print', '(', 'ans', ')']


## 2.3 Saving the Tokenized Train and Validation Artifacts
- Had to use to_parquet in order to preserve the list for the following steps

In [13]:
train_v2.to_parquet('data/processed/train_v2_tokenized.parquet', index=False)
validation_v2.to_parquet('data/processed/validation_v2_tokenized.parquet', index=False)

## 2.4 Verifying the Tokenization Process
- Ensuring that no changes were made during the tokenization process

In [14]:
train_v2_no_tokens = pd.read_csv('data/processed/train_v2.csv')
train_v2_tokens = pd.read_parquet('data/processed/train_v2_tokenized.parquet')

validation_v2_no_tokens = pd.read_csv('data/processed/validation_v2.csv')
validation_v2_tokens = pd.read_parquet('data/processed/validation_v2_tokenized.parquet')

print(f'Train No Tokens: {len(train_v2_no_tokens)} Train Tokens: {len(train_v2_tokens)}')
print(f'Validation No Tokens: {len(validation_v2_no_tokens)} Validation Tokens: {len(validation_v2_tokens)}')

print(f'Testing to ensure there are no empty token sequences (train): {train_v2_tokens[train_v2_tokens['tokens'].apply(len) == 0]}')
print(f'Testing to ensure there are no empty token sequences (validation): {validation_v2_tokens[validation_v2_tokens['tokens'].apply(len) == 0]}')

assert not train_v2_tokens['tokens'].apply(lambda value: isinstance(value, str)).any(), 'There is at least one string in the tokens column (train).'
assert not validation_v2_tokens['tokens'].apply(lambda value: isinstance(value, str)).any(), 'There is at least one string in the tokens column (validation).'

assert not train_v2_tokens['tokens'].apply(len).eq(0).any()
assert not validation_v2_tokens['tokens'].apply(len).eq(0).any()

assert len(train_v2) == len(train_v2_tokens)
assert len(validation_v2) == len(validation_v2_tokens)
assert train_v2["diff"].equals(train_v2_tokens["diff"])
assert validation_v2["diff"].equals(validation_v2_tokens["diff"])
assert train_v2["top_level_label"].equals(train_v2_tokens["top_level_label"])
assert validation_v2["top_level_label"].equals(validation_v2_tokens["top_level_label"])


Train No Tokens: 22809 Train Tokens: 22809
Validation No Tokens: 5703 Validation Tokens: 5703
Testing to ensure there are no empty token sequences (train): Empty DataFrame
Columns: [diff, top_level_label, tokens]
Index: []
Testing to ensure there are no empty token sequences (validation): Empty DataFrame
Columns: [diff, top_level_label, tokens]
Index: []


## Conclusion

I applied the tokenization function to each diff. Diff lines beginning with `+` and `-` were represented using `<ADD>` and `<DELETE>` tokens, respectively. Indentation levels were also represented with explicit tokens, allowing the model to distinguish indentation-only changes from other code changes.

The tokenized datasets were saved in Parquet format to preserve each token sequence as a list rather than converting it into a string. The training set was saved to `data/processed/train_v2_tokenized.parquet`, and the validation set was saved to `data/processed/validation_v2_tokenized.parquet`.